In [4]:
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv("./data/data_AuctionNet/traffic/period-7.csv")
# df = pd.read_csv("./data/data_AuctionNet/traffic/training_data_rlData_folder/period-7-rlData.csv")

# 查看前 5 行
print(df.head())

# 查看字段名、缺失值、数据类型等信息
print(df.info())

   deliveryPeriodIndex  advertiserNumber  advertiserCategoryIndex  budget  \
0                  7.0               0.0                      0.0   750.0   
1                  7.0               1.0                      0.0  1350.0   
2                  7.0               2.0                      0.0  2250.0   
3                  7.0               3.0                      0.0   900.0   
4                  7.0               4.0                      0.0   750.0   

   CPAConstraint  timeStepIndex  remainingBudget  pvIndex    pValue  \
0            8.0            0.0            750.0      0.0  0.005964   
1            7.0            0.0           1350.0      0.0  0.006934   
2            7.0            0.0           2250.0      0.0  0.003792   
3           10.0            0.0            900.0      0.0  0.001558   
4            9.0            0.0            750.0      0.0  0.006401   

   pValueSigma       bid   xi  adSlot  cost  isExposed  conversionAction  \
0          0.0  0.089458  0.0     

In [5]:
import pandas as pd

# 读取 CSV 文件
# df = pd.read_csv("./data/data_AuctionNet/traffic/period-7.csv")
df = pd.read_csv("./data/data_AuctionNet/traffic/training_data_rlData_folder/period-7-rlData.csv")

# 查看前 5 行
print(df.head())

# 查看字段名、缺失值、数据类型等信息
print(df.info())

   deliveryPeriodIndex  advertiserNumber  advertiserCategoryIndex  mt_budget  \
0                  7.0               0.0                      0.0      750.0   
1                  7.0               0.0                      0.0      750.0   
2                  7.0               0.0                      0.0      750.0   
3                  7.0               0.0                      0.0      750.0   
4                  7.0               0.0                      0.0      750.0   

   zt_budget  CPAConstraint  realAllCost  realAllConversion  timeStepIndex  \
0      750.0            8.0   749.997507               50.0            0.0   
1      750.0            8.0   749.997507               50.0            1.0   
2      750.0            8.0   749.997507               50.0            2.0   
3      750.0            8.0   749.997507               50.0            3.0   
4      750.0            8.0   749.997507               50.0            4.0   

                                               sta

In [8]:
import os
import glob
import pandas as pd
from concurrent.futures import ThreadPoolExecutor


# =========================
# 1. 配置参数
# =========================

data_folder = "./data/data_AuctionNet/traffic"

# 是否并行读取多个 CSV 文件
# workers = 1 表示串行
# workers > 1 表示使用多线程并行读取多个文件
workers = 1

# 只读取这两列，节省内存
usecols = ["isExposed", "conversionAction"]


# =========================
# 2. 统计单个 CSV 文件
# =========================

def stat_one_csv(csv_path, chunksize=500_000):
    """
    分块读取单个 CSV 文件，适合大文件。
    """
    file_conversion = 0.0
    file_exposed = 0

    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        chunk["isExposed"] = pd.to_numeric(chunk["isExposed"], errors="coerce")
        chunk["conversionAction"] = pd.to_numeric(chunk["conversionAction"], errors="coerce")

        exposed_chunk = chunk[chunk["isExposed"] == 1]

        file_conversion += exposed_chunk["conversionAction"].sum()
        file_exposed += len(exposed_chunk)

    return {
        "file": os.path.basename(csv_path),
        "exposed_count": file_exposed,
        "conversion_sum": file_conversion,
        "conversionAction_mean": file_conversion / file_exposed if file_exposed > 0 else 0.0,
    }


# =========================
# 3. 统计整个文件夹
# =========================

def calculate_exposed_conversion_mean(data_folder, workers=1):
    csv_paths = sorted(glob.glob(os.path.join(data_folder, "*.csv")))

    if len(csv_paths) == 0:
        raise FileNotFoundError(f"没有在该目录下找到 CSV 文件：{data_folder}")

    if workers == 1:
        results = [stat_one_csv(csv_path) for csv_path in csv_paths]
    else:
        with ThreadPoolExecutor(max_workers=workers) as executor:
            results = list(executor.map(stat_one_csv, csv_paths))

    result_df = pd.DataFrame(results)

    total_conversion = result_df["conversion_sum"].sum()
    total_exposed = result_df["exposed_count"].sum()

    overall_mean = (
        total_conversion / total_exposed
        if total_exposed > 0
        else 0.0
    )

    return result_df, total_exposed, overall_mean


# =========================
# 4. 执行统计
# =========================

result_df, total_exposed, overall_mean = calculate_exposed_conversion_mean(
    data_folder=data_folder,
    workers=workers
)

print(f"overall: exposed_count={total_exposed}, conversionAction_mean={overall_mean:.8f}")

result_df

overall: exposed_count=7985957, conversionAction_mean=0.01072558


,file,exposed_count,conversion_sum,conversionAction_mean
0,period-10.csv,1141137,13132.0,0.011508
1,period-11.csv,1140701,11731.0,0.010284
2,period-12.csv,1140459,12916.0,0.011325
3,period-13.csv,1140445,11145.0,0.009773
4,period-7.csv,1141596,12263.0,0.010742
5,period-8.csv,1140672,12839.0,0.011256
6,period-9.csv,1140947,11628.0,0.010192
